In [1]:
from __future__ import annotations

import json
import os
import random
import re
import time
import uuid
from dataclasses import asdict, dataclass, field
from enum import Enum
from pathlib import Path
from typing import Any, Callable, Optional

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"项目目录: {PROJECT_ROOT.resolve()}")

项目目录: D:\CodeData\Program Coding\Project\Writing_Coach_Agent


In [2]:
class StepStatus(str, Enum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCEEDED = "succeeded"
    FAILED = "failed"


@dataclass
class PlanStep:
    step_id: str
    tool_name: str
    purpose: str
    inputs: dict[str, Any]
    status: StepStatus = StepStatus.PENDING
    result: Any = None
    error: Optional[str] = None


@dataclass
class AgentState:
    task: str
    inputs: dict[str, Any]
    run_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    plan: list[PlanStep] = field(default_factory=list)
    artifacts: dict[str, Any] = field(default_factory=dict)
    trace: list[dict[str, Any]] = field(default_factory=list)
    final_answer: Optional[dict[str, Any]] = None

    def log(self, event: str, **payload: Any) -> None:
        self.trace.append({
            "time": time.strftime("%H:%M:%S"),
            "run_id": self.run_id,
            "event": event,
            **payload,
        })


class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., Any]] = {}

    def register(self, name: str, func: Callable[..., Any]) -> None:
        if name in self._tools:
            raise ValueError(f"工具已注册: {name}")
        self._tools[name] = func

    def call(self, name: str, **kwargs: Any) -> Any:
        if name not in self._tools:
            raise KeyError(f"未知工具: {name}")
        return self._tools[name](**kwargs)

    @property
    def names(self) -> list[str]:
        return sorted(self._tools)

In [3]:
class RetryableToolError(RuntimeError):
    """临时性错误：重试后可能恢复。"""


class FatalToolError(RuntimeError):
    """不可恢复错误：应停止或重新规划。"""


@dataclass
class Memory:
    working: dict[str, Any] = field(default_factory=dict)
    episodic: list[dict[str, Any]] = field(default_factory=list)

    def remember_result(self, step: PlanStep) -> None:
        self.working[step.step_id] = step.result
        self.episodic.append({"step_id": step.step_id, "tool": step.tool_name, "status": step.status.value})


class CheckpointStore:
    def __init__(self, folder: Path) -> None:
        self.folder = folder
        self.folder.mkdir(parents=True, exist_ok=True)

    def save(self, state: AgentState, memory: Memory) -> Path:
        path = self.folder / f"{state.run_id}.json"
        payload = {
            "run_id": state.run_id,
            "task": state.task,
            "inputs": state.inputs,
            "artifacts": state.artifacts,
            "trace": state.trace,
            "memory": asdict(memory),
            "plan": [asdict(step) for step in state.plan],
        }
        path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
        return path

In [4]:
class FlakyEvidenceTool:
    def __init__(self, fail_times: int = 1) -> None:
        self.remaining_failures = fail_times

    def __call__(self, essay: str) -> dict[str, Any]:
        if self.remaining_failures > 0:
            self.remaining_failures -= 1
            raise RetryableToolError("模拟的模型服务 503")
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", essay) if s.strip()]
        return {"evidence": sentences[:2], "quality": "ok" if len(sentences) >= 2 else "weak"}


def validate_input_tool(essay: str) -> dict[str, Any]:
    if not isinstance(essay, str) or not essay.strip():
        raise FatalToolError("作文为空")
    return {"essay": re.sub(r"\s+", " ", essay).strip()}


def score_tool(essay: str) -> dict[str, Any]:
    words = re.findall(r"\b[A-Za-z']+\b", essay)
    markers = sum(bool(re.search(rf"\b{x}\b", essay, re.I)) for x in ["because", "however", "therefore", "example"])
    return {"language": round(min(5, 1.5 + len(words) / 45), 2), "argumentation": round(min(5, 1.2 + markers * 0.75), 2)}


def report_tool(scores: dict[str, Any], evidence: dict[str, Any]) -> dict[str, Any]:
    return {"scores": scores, "evidence": evidence["evidence"], "status": "completed"}

In [5]:
@dataclass
class Reflection:
    action: str  # retry / replan / continue / stop
    reason: str


class Reflector:
    def review_failure(self, step: PlanStep, attempt: int, max_retries: int) -> Reflection:
        if step.error and "RetryableToolError" in step.error and attempt < max_retries:
            return Reflection("retry", "临时性错误且仍有重试预算")
        if step.tool_name == "extract_evidence":
            return Reflection("replan", "证据工具不可用，改用轻量降级工具")
        return Reflection("stop", "错误不可恢复或超出预算")

    def review_result(self, step: PlanStep) -> Reflection:
        if step.tool_name == "extract_evidence" and step.result.get("quality") == "weak":
            return Reflection("continue", "证据较弱，但仍可生成带风险提示的报告")
        return Reflection("continue", "结果通过基本检查")


def should_stop(state: AgentState, total_steps: int, max_steps: int = 10) -> tuple[bool, str]:
    if len([x for x in state.trace if x["event"] == "tool_started"]) >= max_steps:
        return True, "达到最大工具调用次数"
    if state.final_answer is not None:
        return True, "已产生最终答案"
    if all(s.status == StepStatus.SUCCEEDED for s in state.plan[:total_steps]):
        return False, "当前计划已完成，等待组装答案"
    return False, "继续执行"

In [6]:
import importlib.util
import os
import subprocess
import sys

PACKAGE_MAP = {
    "numpy": "numpy>=1.24,<2.3",
    "pandas": "pandas>=2.0,<2.4",
}

RUN_LOCAL_MODEL = True
RUN_LOCAL_MODEL = RUN_LOCAL_MODEL or os.getenv("WRITING_COACH_USE_LOCAL_MODEL", "0") == "1"
if RUN_LOCAL_MODEL:
    PACKAGE_MAP.update({
        "torch": "torch>=2.1,<2.8",
        "transformers": "transformers>=4.45,<5",
        "accelerate": "accelerate>=0.29,<2",
    })
missing = [requirement for module, requirement in PACKAGE_MAP.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("安装缺失依赖:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--index-url", "https://pypi.org/simple", *missing])
else:
    print("✅ 第 3 课依赖已就绪")

✅ 第 3 课依赖已就绪


In [7]:
from typing import Protocol

def extract_json(text: str) -> dict[str, Any]:
    """兼容纯 JSON 与 ```json fenced output。"""
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.I)
    try:
        value = json.loads(cleaned)
        if isinstance(value, dict):
            return value
    except json.JSONDecodeError:
        pass
    decoder = json.JSONDecoder()
    for start, char in enumerate(cleaned):
        if char != "{":
            continue
        try:
            value, _ = decoder.raw_decode(cleaned[start:])
            if isinstance(value, dict):
                return value
        except json.JSONDecodeError:
            continue
    raise ValueError(f"模型没有返回可解析 JSON: {cleaned[:160]}")

In [8]:
class JSONBackend(Protocol):
    name: str
    def generate_json(self, system: str, user: str, schema: dict[str, Any]) -> dict[str, Any]: ...

In [9]:
def split_sentences(essay: str) -> list[str]:
    return [x.strip() for x in re.split(r"(?<=[.!?])\s+", essay.strip()) if x.strip()]

def text_analysis(essay: str, rubric_path: Path) -> dict[str, Any]:
    sentences = split_sentences(essay)
    return {
        "word_count": len(re.findall(r"\b[A-Za-z']+\b", essay)),
        "sentences": [{"id": i, "text": sentence} for i, sentence in enumerate(sentences, 1)],
    }

def rubric_lookup(essay: str, rubric_path: Path) -> dict[str, Any]:
    return json.loads(rubric_path.read_text(encoding="utf-8"))

def evidence_locator(essay: str, rubric_path: Path) -> dict[str, Any]:
    patterns = {
        "claim": r"\b(should|must|believe|opinion)\b",
        "reason": r"\b(because|since|reason)\b",
        "example": r"\b(for example|for instance|such as)\b",
        "counterargument": r"\b(however|although|some people)\b",
        "conclusion": r"\b(therefore|in conclusion|to conclude)\b",
    }
    rows = []
    for i, sentence in enumerate(split_sentences(essay), 1):
        labels = [name for name, pattern in patterns.items() if re.search(pattern, sentence, re.I)]
        rows.append({"sentence_id": i, "labels": labels, "text": sentence})
    return {"sentence_evidence": rows}

AI_TOOLS = {
    "text_analysis": text_analysis,
    "rubric_lookup": rubric_lookup,
    "evidence_locator": evidence_locator,
}
display(pd.DataFrame([
    {"tool": name, "role": role} for name, role in {
        "text_analysis": "Product data：客观文本事实",
        "rubric_lookup": "Grounding data：评分标准",
        "evidence_locator": "Evidence：原文句子编号",
    }.items()
]))

,tool,role
0,text_analysis,Product data：客观文本事实
1,rubric_lookup,Grounding data：评分标准
2,evidence_locator,Evidence：原文句子编号


In [10]:
class LocalQwenBackend:
    """惰性加载：同一 Kernel 只加载一次，后续模型调用复用内存中的对象。"""
    def __init__(self, model_id: str = "Qwen/Qwen2.5-0.5B-Instruct") -> None:
        self.model_id = model_id
        self.name = f"local-open-source:{model_id}"
        self.tokenizer = None
        self.model = None
        self.load_seconds = None

    def load(self) -> float:
        if self.model is not None:
            return 0.0
        started = time.perf_counter()
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id, dtype="auto", low_cpu_mem_usage=True
        )
        self.model.eval()
        self.model.generation_config.temperature = None
        self.model.generation_config.top_p = None
        self.model.generation_config.top_k = None
        self.load_seconds = time.perf_counter() - started
        return self.load_seconds

    def _generate_once(self, system: str, user: str, schema: dict[str, Any],
                       repair: str = "") -> dict[str, Any]:
        self.load()
        import torch
        task = schema.get("task")
        messages = [
            {"role": "system", "content": system +
             " Return exactly one valid JSON object, no Markdown. Include every contract field."},
            {"role": "user", "content": user + "\nJSON contract: " +
             json.dumps(schema, ensure_ascii=False) + repair},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt")
        started = time.perf_counter()
        with torch.inference_mode():
            # Planner/Reflector 只需要短 JSON。严格限制 token 预算，
            # 避免 CPU 课堂环境中小模型反复生成合同示例。
            token_budget = {"plan": 48, "reflect": 32}.get(task, 64)
            output = self.model.generate(
                **inputs, max_new_tokens=token_budget, do_sample=False,
                repetition_penalty=1.05, pad_token_id=self.tokenizer.eos_token_id,
            )
        generated = output[0][inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(generated, skip_special_tokens=True)
        result = extract_json(text)
        result["_inference_seconds"] = round(time.perf_counter() - started, 2)
        return result
        
    def _generate_text(self, user: str, instruction: str, max_new_tokens: int = 48) -> str:
        """让小模型生成一个短字段；Python 只封装结构，不替模型写评分内容。"""
        import torch
        messages = [
            {"role": "system", "content": instruction + " Return one concise sentence only."},
            {"role": "user", "content": user},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt")
        with torch.inference_mode():
            output = self.model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                repetition_penalty=1.05, pad_token_id=self.tokenizer.eos_token_id,
            )
        text = self.tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        text = text.splitlines()[0].strip().strip('"')
        if not text:
            raise ValueError("local model returned an empty coaching field")
        return text
    
    def _select_score(self, user: str, dimension: str) -> float:
        """直接比较本地模型对 1..5 的 next-token 概率，结构上不会漏 score。"""
        import torch
        messages = [
            {"role": "system", "content":
             "You are a rubric scorer. Choose one integer from 1 (weak) to 5 (strong)."},
            {"role": "user", "content": user +
             f"\nDimension: {dimension}. Answer with exactly one digit. Score:"},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt")
        with torch.inference_mode():
            logits = self.model(**inputs).logits[0, -1]
        candidates = {}
        for score in range(1, 6):
            token_ids = set()
            for form in (str(score), " " + str(score)):
                encoded = self.tokenizer.encode(form, add_special_tokens=False)
                if encoded:
                    token_ids.add(encoded[0])
            candidates[score] = max(float(logits[token_id]) for token_id in token_ids)
        return float(max(candidates, key=candidates.get))
    
    def _generate_plan(self, user: str) -> dict[str, Any]:
        """用模型 logits 在白名单工具中选择，避免短 JSON 被截断。"""
        import torch
        options = {1: "text_analysis", 2: "rubric_lookup", 3: "evidence_locator"}
        messages = [
            {"role": "system", "content": "Choose the most useful first tool for this Writing Coach task."},
            {"role": "user", "content": user +
             "\n1=text_analysis, 2=rubric_lookup, 3=evidence_locator. Answer one digit only:"},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt")
        with torch.inference_mode():
            logits = self.model(**inputs).logits[0, -1]
        strengths = {}
        for number in options:
            ids = self.tokenizer.encode(str(number), add_special_tokens=False)
            if ids:
                strengths[number] = float(logits[ids[0]])
        selected = options[max(strengths, key=strengths.get)]
        reason = self._generate_text(
            user, f"Explain briefly why {selected} is useful for this exact task.", 16)
        return {"goal": "model-selected grounded diagnosis",
                "steps": [{"tool": selected, "reason": reason}],
                "_structured_from_local_model_fields": True}
    
    def _generate_report(self, user: str) -> dict[str, Any]:
        """用本地模型的原子输出组装合同，避免 0.5B 长 JSON 截断或改字段名。"""
        language_score = self._select_score(user, "language clarity, grammar, cohesion")
        argument_score = self._select_score(user, "claim, reasons, evidence, counterargument")
        # 用一次短生成同时得到证据化诊断和修订动作，避免 CPU 上重复解码。
        # 内容仍由模型生成；Python 只是在稳定 JSON 合同中复用该字段。
        coaching = self._generate_text(
            user,
            "In one short sentence, diagnose the most important language or argument issue and give one revision action without rewriting the essay.",
            24,
        )
        evidence_id = self._select_evidence_id(user)
        return {
            "summary": coaching,
            "scores": {
                "language": {"score": language_score, "reason": coaching, "evidence_ids": [evidence_id]},
                "argumentation": {"score": argument_score, "reason": coaching, "evidence_ids": [evidence_id]},
            },
            "priorities": [{"issue": coaching, "evidence_id": evidence_id,
                            "action": coaching, "example": "For example, ... This shows that ..."}],
            "revision_plan": [coaching],
            "highlights": [{"sentence_id": evidence_id, "label": "needs_evidence", "reason": coaching}],
            "_structured_from_local_model_fields": True,
        }
    
    def _select_evidence_id(self, user: str) -> int:
        """让模型在真实句号中选证据，不把第 1 句写死。"""
        import torch
        try:
            payload = json.loads(user)
            rows = payload.get("facts", {}).get("evidence_locator", {}).get("sentence_evidence", [])
            valid_ids = [int(row["sentence_id"]) for row in rows][:9]
        except Exception:
            valid_ids = list(range(1, min(len(split_sentences(user)), 9) + 1))
        if not valid_ids:
            return 1
        messages = [
            {"role": "system", "content": "Choose the single sentence ID that most needs evidence or explanation."},
            {"role": "user", "content": user + f"\nValid IDs: {valid_ids}. Answer one digit only. ID:"},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt")
        with torch.inference_mode():
            logits = self.model(**inputs).logits[0, -1]
        strengths = {}
        for sentence_id in valid_ids:
            token_ids = self.tokenizer.encode(str(sentence_id), add_special_tokens=False)
            if token_ids:
                strengths[sentence_id] = float(logits[token_ids[0]])
        return max(strengths, key=strengths.get) if strengths else valid_ids[0]
    
    def generate_json(self, system: str, user: str, schema: dict[str, Any]) -> dict[str, Any]:
        if schema.get("task") == "plan":
            self.load()
            return self._generate_plan(user)
        if schema.get("task") == "report":
            self.load()
            return self._generate_report(user)
        if schema.get("task") == "reflect":
            self.load()
            quality = self._select_score(user, "grounding, rubric use, actionability, and no full-essay rewriting")
            return {"decision": "accept" if quality >= 3 else "revise",
                    "reason": f"Local model reflection score: {quality:.0f}/5",
                    "repair": "Improve grounding and actionability." if quality < 3 else ""}
        result = self._generate_once(system, user, schema)
        return result

In [11]:
class ClassroomBackend:
    """快速、离线、可重复的课堂降级后端"""
    name = "classroom-fallback-rules (NOT AI)"
    def generate_json(self, system: str, user: str, schema: dict[str, Any]) -> dict[str, Any]:
        task = schema["task"]
        if task == "plan":
            return {"goal": "按量表和作文证据生成诊断", "steps": [
                {"tool": "text_analysis", "reason": "复用第 2 课的文本事实"},
                {"tool": "rubric_lookup", "reason": "读取评分标准"},
                {"tool": "evidence_locator", "reason": "定位可引用句子"},
            ]}
        if task == "report":
            return {
                "summary": "立场可识别，但论证需要更具体的证据与解释。",
                "scores": {
                    "language": {"score": 2.8, "reason": "表达基本可理解，仍有语法和衔接问题。", "evidence_ids": [1]},
                    "argumentation": {"score": 2.4, "reason": "有主张，但理由与例证展开不足。", "evidence_ids": [1, 2]},
                },
                "priorities": [{"issue": "论证展开不足", "evidence_id": 2,
                    "action": "在该理由后补充一个具体例子，并解释例子如何支持主张。",
                    "example": "For example, ... This shows that ..."}],
                "revision_plan": ["补充一个具体例子", "解释例子与主张的关系", "检查句间衔接"],
                "highlights": [{"sentence_id": 2, "label": "needs_evidence", "reason": "理由后缺少证据"}],
            }
        if task == "reflect":
            return {"decision": "accept", "reason": "字段完整，建议包含证据编号与可执行动作。", "repair": ""}
        raise ValueError(f"未知任务: {task}")

In [12]:
class SafeBackend:
    """真实模型失败时自动降级，并在结果中显式记录原因。"""
    def __init__(self, primary: JSONBackend, fallback: JSONBackend) -> None:
        self.primary, self.fallback = primary, fallback
        self.name = primary.name
        self.degraded = False
        self.last_error = None

    def generate_json(self, system: str, user: str, schema: dict[str, Any]) -> dict[str, Any]:
        if self.degraded:
            return self.fallback.generate_json(system, user, schema)
        try:
            result = self.primary.generate_json(system, user, schema)
            self.name = self.primary.name
            return result
        except Exception as exc:
            self.force_fallback(f"{type(exc).__name__}: {exc}")
            return self.fallback.generate_json(system, user, schema)

    def force_fallback(self, error: str) -> None:
        self.degraded = True
        self.last_error = error
        self.name = self.fallback.name
        print("⚠️ 模型输出未通过报告合同，切换课堂降级后端:", error)

    def reset(self) -> None:
        """新一次独立运行重新尝试本地模型，避免一次失败永久污染后续轮次。"""
        self.degraded = False
        self.last_error = None
        self.name = self.primary.name

In [13]:
local_model = LocalQwenBackend(os.getenv("WRITING_COACH_MODEL", "Qwen/Qwen2.5-0.5B-Instruct"))
backend: JSONBackend
if RUN_LOCAL_MODEL:
    backend = SafeBackend(local_model, ClassroomBackend())
    seconds = local_model.load()
    print(f"✅ 模型已加载；本次耗时 {seconds:.2f}s；同一 Kernel 后续加载耗时为 0s")
else:
    backend = ClassroomBackend()
    print("快速课堂模式：未加载模型。要启用 AI，请把第一个代码单元格的 RUN_LOCAL_MODEL 改为 True 后从头运行。")
print("当前后端:", backend.name)

✅ 模型已加载；本次耗时 25.65s；同一 Kernel 后续加载耗时为 0s
当前后端: local-open-source:Qwen/Qwen2.5-0.5B-Instruct


In [14]:
PLAN_SCHEMA = {"task": "plan", "goal": "string", "steps": [{"tool": "allowed tool", "reason": "string"}]}
REPORT_SCHEMA = {"task": "report", "summary": "string", "scores": {
    "language": {"score": "1..5", "reason": "string", "evidence_ids": [1]},
    "argumentation": {"score": "1..5", "reason": "string", "evidence_ids": [1]}},
    "priorities": [{"issue": "string", "evidence_id": 1, "action": "string", "example": "short fragment"}],
    "revision_plan": ["ordered action"],
    "highlights": [{"sentence_id": 1, "label": "strength|needs_evidence|language|counterargument", "reason": "string"}]}
REFLECT_SCHEMA = {"task": "reflect", "decision": "accept|revise", "reason": "string", "repair": "string"}


@dataclass
class AIRun:
    prompt: str
    essay: str
    run_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    stage: str = "started"
    plan: list[dict[str, str]] = field(default_factory=list)
    artifacts: dict[str, Any] = field(default_factory=dict)
    report: dict[str, Any] | None = None
    trace: list[dict[str, Any]] = field(default_factory=list)

    def log(self, event: str, **details: Any) -> None:
        self.trace.append({"time": time.strftime("%H:%M:%S"), "event": event, **details})


class SimulatedCrash(RuntimeError):
    pass

In [15]:
class RecoverableWritingCoach:
    def __init__(self, backend: JSONBackend, rubric_path: Path, checkpoint_dir: Path,
                 max_retries: int = 2) -> None:
        self.backend = backend
        self.rubric_path = rubric_path
        self.checkpoint_dir = checkpoint_dir
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.max_retries = max_retries

    def _checkpoint_path(self, run_id: str) -> Path:
        return self.checkpoint_dir / f"{run_id}.json"

    def save(self, run: AIRun) -> Path:
        path = self._checkpoint_path(run.run_id)
        path.write_text(json.dumps(asdict(run), ensure_ascii=False, indent=2), encoding="utf-8")
        run.log("checkpoint_saved", stage=run.stage, path=str(path))
        # 再保存一次，把 checkpoint_saved 本身也写入快照。
        path.write_text(json.dumps(asdict(run), ensure_ascii=False, indent=2), encoding="utf-8")
        return path

    def load(self, run_id: str) -> AIRun:
        payload = json.loads(self._checkpoint_path(run_id).read_text(encoding="utf-8"))
        run = AIRun(**payload)
        run.log("checkpoint_loaded", stage=run.stage)
        return run
    
    def ask(self, run: AIRun, stage: str, system: str, user: str, schema: dict[str, Any]) -> dict[str, Any]:
        for attempt in range(1, self.max_retries + 2):
            run.log("llm_started", stage=stage, attempt=attempt, backend=self.backend.name)
            try:
                result = self.backend.generate_json(system, user, schema)
                run.log("llm_succeeded", stage=stage, attempt=attempt, backend=self.backend.name)
                return result
            except (TimeoutError, ConnectionError, RuntimeError, ValueError) as exc:
                run.log("llm_failed", stage=stage, attempt=attempt, error=f"{type(exc).__name__}: {exc}")
                self.save(run)
                if attempt > self.max_retries:
                    raise
                delay = min(0.25 * 2 ** (attempt - 1), 1.0)
                run.log("retry_scheduled", stage=stage, delay_seconds=delay)
                time.sleep(delay)
        raise RuntimeError("unreachable")

    def _validate_plan(self, raw: dict[str, Any]) -> list[dict[str, str]]:
        valid, seen = [], set()
        steps = raw.get("steps", [])
        if not steps and raw.get("tool"):
            steps = [raw]
        for step in steps:
            tool = step.get("tool")
            if tool in AI_TOOLS and tool not in seen:
                valid.append({"tool": tool, "reason": str(step.get("reason", ""))})
                seen.add(tool)
        if not valid:
            raise ValueError("Planner 没有选择合法工具")
        # Planner 保留自主选择；Executor 只补齐不可放弃的 grounding 底线。
        # 这和 Web Agent 强制登录检查、金融 Agent 强制风险检查是同一种设计。
        for required, reason in [
            ("rubric_lookup", "Executor guardrail：评分前必须读取 Rubric"),
            ("evidence_locator", "Executor guardrail：建议必须引用真实句号"),
        ]:
            if required not in seen:
                valid.append({"tool": required, "reason": reason})
                seen.add(required)
        return valid
    
    def _validate_report(self, report: dict[str, Any], sentence_count: int) -> None:
        if not isinstance(report.get("scores"), dict):
            raise ValueError("报告缺少 scores 对象")
        for dimension in ("language", "argumentation"):
            item = report["scores"].get(dimension)
            if not isinstance(item, dict):
                raise ValueError(f"scores 缺少 {dimension}")
            score = float(item.get("score"))
            if not 1 <= score <= 5:
                raise ValueError(f"{dimension} 分数越界: {score}")
            if not item.get("reason") or not item.get("evidence_ids"):
                raise ValueError(f"{dimension} 缺少 reason 或 evidence_ids")
        for item in report.get("highlights", []):
            if not 1 <= int(item["sentence_id"]) <= max(sentence_count, 1):
                raise ValueError("高亮句子编号越界")
    
    def run(self, prompt: str | None = None, essay: str | None = None,
            resume_run_id: str | None = None, stop_after: str | None = None) -> AIRun:
        if isinstance(self.backend, SafeBackend):
            self.backend.reset()
        run = self.load(resume_run_id) if resume_run_id else AIRun(prompt=prompt or "", essay=essay or "")
        if not run.essay.strip():
            raise ValueError("作文不能为空")

        if run.stage == "started":
            raw_plan = self.ask(run, "planning",
                "You are the planner of a Writing Coach Agent. Select only tools needed to ground a rubric-based diagnosis.",
                f"Prompt: {run.prompt}\nEssay excerpt: {run.essay[:600]}\nAvailable tools: {list(AI_TOOLS)}", PLAN_SCHEMA)
            run.plan = self._validate_plan(raw_plan)
            run.stage = "planned"
            run.log("plan_created", plan=run.plan)
            self.save(run)

        if run.stage == "planned":
            for step_no, step in enumerate(run.plan, 1):
                tool = step["tool"]
                if tool in run.artifacts:
                    run.log("tool_skipped_after_resume", tool=tool)
                    continue
                run.log("tool_started", step=step_no, tool=tool)
                run.artifacts[tool] = AI_TOOLS[tool](run.essay, self.rubric_path)
                run.log("tool_succeeded", step=step_no, tool=tool)
                self.save(run)
            run.stage = "tools_completed"
            self.save(run)
            if stop_after == "tools_completed":
                raise SimulatedCrash(f"模拟 Kernel 中断；run_id={run.run_id}")
        
        if run.stage == "tools_completed":
            context = json.dumps({"prompt": run.prompt, "essay": run.essay, "facts": run.artifacts}, ensure_ascii=False)
            report = self.ask(run, "coaching",
                "You are a supportive Writing Coach. Score with the rubric and cited sentence IDs. Give prioritized, actionable advice; do not rewrite the whole essay.",
                context, REPORT_SCHEMA)
            try:
                self._validate_report(report, len(split_sentences(run.essay)))
            except (KeyError, TypeError, ValueError) as exc:
                error = f"{type(exc).__name__}: {exc}"
                run.log("schema_validation_failed", stage="coaching", error=error,
                        raw_output=report)
                if isinstance(self.backend, SafeBackend):
                    self.backend.force_fallback(f"coaching: {error}")
                    report = self.backend.generate_json(
                        "Classroom fallback report", context, REPORT_SCHEMA)
                    self._validate_report(report, len(split_sentences(run.essay)))
                else:
                    raise ValueError(f"Coach 报告不符合合同: {error}") from exc
            reflection = self.ask(run, "reflection",
                "You are an independent reflector. Check grounding, rubric use, actionability, and over-writing risk.",
                json.dumps({"essay": run.essay, "report": report}, ensure_ascii=False), REFLECT_SCHEMA)
            run.log("reflection_completed", **reflection)

            if reflection.get("decision") == "revise":
                # Reflector 不只记一条 warning，而是把修复指令回送给 Coach，形成真正的反思闭环。
                repair = str(reflection.get("repair", ""))
                run.log("report_repair_started", repair=repair)
                repaired_context = context + "\nReflector repair instruction: " + repair
                repaired = self.ask(run, "coaching_repair",
                    "Revise the previous coaching report according to the independent reflector. Keep valid sentence IDs and do not rewrite the essay.",
                    repaired_context, REPORT_SCHEMA)
                self._validate_report(repaired, len(split_sentences(run.essay)))
                report = repaired
                run.log("report_repair_succeeded")
            report["model_backend"] = self.backend.name
            report["degraded"] = getattr(self.backend, "degraded", False) or "NOT AI" in self.backend.name
            run.report = report
            run.stage = "completed"
            run.log("run_finished", success=True)
            self.save(run)
        return run
        

In [16]:
rubric_path = DATA_DIR / "rubric.jsonl"
coach = RecoverableWritingCoach(backend, rubric_path, OUTPUT_DIR / "recoverable_agent")
sample = json.loads((DATA_DIR / "essays.jsonl").read_text(encoding="utf-8").splitlines()[0])
prompt = sample.get("prompt", "Write an argumentative essay.")

try:
    coach.run(prompt, sample["essay"], stop_after="tools_completed")
except SimulatedCrash as exc:
    interrupted_run_id = str(exc).split("run_id=")[-1]
    print("预期中的中断:", exc)

before = coach.load(interrupted_run_id)
tool_calls_before = sum(x["event"] == "tool_started" for x in before.trace)
recovered = coach.run(resume_run_id=interrupted_run_id)
tool_calls_after = sum(x["event"] == "tool_started" for x in recovered.trace)

print("恢复前 stage:", before.stage)
print("恢复后 stage:", recovered.stage)
print("恢复前/后累计工具调用:", tool_calls_before, tool_calls_after)
assert recovered.report is not None and recovered.stage == "completed"
assert tool_calls_before == tool_calls_after
print("✅ 从 Checkpoint 恢复成功，已完成的工具没有重复执行")

预期中的中断: 模拟 Kernel 中断；run_id=85363790
恢复前 stage: tools_completed
恢复后 stage: completed
恢复前/后累计工具调用: 3 3
✅ 从 Checkpoint 恢复成功，已完成的工具没有重复执行


In [17]:
display(pd.DataFrame(recovered.plan))
print(json.dumps(recovered.report, ensure_ascii=False, indent=2))
trace_df = pd.DataFrame(recovered.trace).fillna("")
display(trace_df)

summary = {
    "stage": recovered.stage,
    "llm_calls": int((trace_df["event"] == "llm_started").sum()),
    "tool_calls": int((trace_df["event"] == "tool_started").sum()),
    "checkpoints": int((trace_df["event"] == "checkpoint_saved").sum()),
    "resumed": bool((trace_df["event"] == "checkpoint_loaded").any()),
    "backend": recovered.report["model_backend"],
    "degraded": recovered.report["degraded"],
}
display(pd.DataFrame([summary]))
assert summary["resumed"] and summary["checkpoints"] >= 2

,tool,reason
0,text_analysis,Social media use in schools can be detrimental...
1,rubric_lookup,Executor guardrail：评分前必须读取 Rubric
2,evidence_locator,Executor guardrail：建议必须引用真实句号


{
  "summary": "Limit social media in class while allowing supervised educational use, addressing distraction and support for group projects.",
  "scores": {
    "language": {
      "score": 3.0,
      "reason": "Limit social media in class while allowing supervised educational use, addressing distraction and support for group projects.",
      "evidence_ids": [
        1
      ]
    },
    "argumentation": {
      "score": 3.0,
      "reason": "Limit social media in class while allowing supervised educational use, addressing distraction and support for group projects.",
      "evidence_ids": [
        1
      ]
    }
  },
  "priorities": [
    {
      "issue": "Limit social media in class while allowing supervised educational use, addressing distraction and support for group projects.",
      "evidence_id": 1,
      "action": "Limit social media in class while allowing supervised educational use, addressing distraction and support for group projects.",
      "example": "For example, .

,time,event,stage,attempt,backend,plan,path,step,tool,decision,reason,repair,success
0,19:12:43,llm_started,planning,1.0,local-open-source:Qwen/Qwen2.5-0.5B-Instruct,,,,,,,,
1,19:12:47,llm_succeeded,planning,1.0,local-open-source:Qwen/Qwen2.5-0.5B-Instruct,,,,,,,,
2,19:12:47,plan_created,,,,"[{'tool': 'text_analysis', 'reason': 'Social m...",,,,,,,
3,19:12:47,checkpoint_saved,planned,,,,d:\CodeData\Program Coding\Project\Writing_Coa...,,,,,,
4,19:12:47,tool_started,,,,,,1.0,text_analysis,,,,
5,19:12:47,tool_succeeded,,,,,,1.0,text_analysis,,,,
6,19:12:47,checkpoint_saved,planned,,,,d:\CodeData\Program Coding\Project\Writing_Coa...,,,,,,
7,19:12:47,tool_started,,,,,,2.0,rubric_lookup,,,,
8,19:12:47,tool_succeeded,,,,,,2.0,rubric_lookup,,,,
9,19:12:47,checkpoint_saved,planned,,,,d:\CodeData\Program Coding\Project\Writing_Coa...,,,,,,


,stage,llm_calls,tool_calls,checkpoints,resumed,backend,degraded
0,completed,3,3,6,True,local-open-source:Qwen/Qwen2.5-0.5B-Instruct,False


In [18]:
# 只调用 Planner，不运行后续评分；这样可以单独观察“模型如何选工具”。
planning_cases = [
    {
        "case": "rubric_feedback",
        "task": "Score this essay with the supplied rubric and cite evidence.",
        "essay": sample["essay"],
    },
    {
        "case": "evidence_only",
        "task": "Find which sentence most needs concrete evidence; do not score.",
        "essay": "School uniforms are useful. They help students because everyone looks similar.",
    },
]
planning_rows = []
for case in planning_cases:
    probe = AIRun(prompt=case["task"], essay=case["essay"])
    raw = coach.ask(
        probe,
        "planning_probe",
        "Choose only the tools needed for this request. Never invent a tool.",
        f"Task: {case['task']}\nEssay excerpt: {case['essay'][:400]}\n"
        f"Available tools: {list(AI_TOOLS)}",
        PLAN_SCHEMA,
    )
    try:
        chosen = coach._validate_plan(raw)
        status = "valid"
    except ValueError as exc:
        chosen, status = [], f"rejected: {exc}"
    planning_rows.append({
        "case": case["case"],
        "model_backend": backend.name,
        "selected_tools": [step["tool"] for step in chosen],
        "model_reasons": [step["reason"] for step in chosen],
        "guardrail_status": status,
    })

planning_audit = pd.DataFrame(planning_rows)
display(planning_audit)
assert all(value == "valid" for value in planning_audit["guardrail_status"])
print("✅ Planner 决定来自模型，执行权限仍由 Tool whitelist 控制")

,case,model_backend,selected_tools,model_reasons,guardrail_status
0,rubric_feedback,local-open-source:Qwen/Qwen2.5-0.5B-Instruct,"[text_analysis, rubric_lookup, evidence_locator]",[Text analysis helps identify distracting feat...,valid
1,evidence_only,local-open-source:Qwen/Qwen2.5-0.5B-Instruct,"[rubric_lookup, evidence_locator]",[Use the text analysis tool to identify the ma...,valid


✅ Planner 决定来自模型，执行权限仍由 Tool whitelist 控制


In [20]:
class MissingScoresBackend:
    name = "malformed-model-for-test"
    def generate_json(self, system, user, schema):
        if schema["task"] == "plan":
            return {"tool": "evidence_locator", "reason": "测试单步 AI Plan 格式"}
        if schema["task"] == "report":
            return {"summary": "故意缺少 scores 的模型输出"}
        return {"decision": "accept", "reason": "test", "repair": ""}

schema_guard_backend = SafeBackend(MissingScoresBackend(), ClassroomBackend())
schema_guard_coach = RecoverableWritingCoach(
    schema_guard_backend, rubric_path, OUTPUT_DIR / "schema_guard_test")
schema_guard_run = schema_guard_coach.run(prompt, sample["essay"])
schema_events = [x for x in schema_guard_run.trace if x["event"] == "schema_validation_failed"]

print("Schema 失败事件:", schema_events[0]["error"])
print("最终后端:", schema_guard_run.report["model_backend"])
print("最终 scores 字段:", list(schema_guard_run.report["scores"]))
assert schema_events and schema_guard_run.report["degraded"] is True
assert set(schema_guard_run.report["scores"]) == {"language", "argumentation"}
print("✅ 缺少 scores 的模型输出已被拦截并安全恢复，不再产生 KeyError")

⚠️ 模型输出未通过报告合同，切换课堂降级后端: coaching: ValueError: 报告缺少 scores 对象
Schema 失败事件: ValueError: 报告缺少 scores 对象
最终后端: classroom-fallback-rules (NOT AI)
最终 scores 字段: ['language', 'argumentation']
✅ 缺少 scores 的模型输出已被拦截并安全恢复，不再产生 KeyError


In [21]:
def trace_metrics(run: AIRun) -> dict[str, Any]:
    """将事件日志折叠成一行运行指标。
    """
    names = [event["event"] for event in run.trace]
    return {
        "run_id": run.run_id,
        "stage": run.stage,
        "llm_calls": names.count("llm_started"),
        "tool_calls": names.count("tool_started"),
        "retries": names.count("retry_scheduled"),
        "resume_loads": names.count("checkpoint_loaded"),
        "schema_failures": names.count("schema_validation_failed"),
        "checkpoints": names.count("checkpoint_saved"),
        "backend": run.report.get("model_backend") if run.report else "not-finished",
        "degraded": run.report.get("degraded") if run.report else None,
    }

operations_dashboard = pd.DataFrame([
    trace_metrics(recovered),
    trace_metrics(schema_guard_run),
])
display(operations_dashboard)
print("读表方法：第一行验证中断恢复；第二行验证 Schema 失败后的显式降级。")

,run_id,stage,llm_calls,tool_calls,retries,resume_loads,schema_failures,checkpoints,backend,degraded
0,85363790,completed,3,3,0,1,0,6,local-open-source:Qwen/Qwen2.5-0.5B-Instruct,False
1,2e97accc,completed,3,2,0,0,1,5,classroom-fallback-rules (NOT AI),True


读表方法：第一行验证中断恢复；第二行验证 Schema 失败后的显式降级。
